# 05. Vanilla Dirichlet Belief Simulation

This notebook simulates users' beliefs over level-1 video categories with a vanilla Dirichlet update.

Final output:

```text
KuaiRec 2.0/data/processed/valid_user_belief.parquet
```

The saved table is long format:

| Column | Meaning |
|---|---|
| `user_id` | User ID. |
| `burst_id` | Dense activity burst. |
| `session` | User session ID. |
| `i_cat_level1_id` | Level-1 category ID. |
| `belief_prob` | Belief probability at the start of the session. |

## Belief Initialization

Let `K` be the number of observed level-1 categories and let `R_{ibt,k}` be the count of category `k` videos shown to user `i` in burst `b`, session `t`.

For a user's first observed burst:

```text
alpha_{i,b_first,start,k} = PRIOR_ALPHA
```

For later bursts, the prior carries over previous-burst exposure history:

```text
alpha_{i,b,start,k}
= PRIOR_ALPHA + sum_{b' < b} sum_t R_{ib'tk}
```

Within a burst:

```text
belief before session t:
    m_{ibt,k} = alpha_{ibt,k} / sum_j alpha_{ibt,j}

update after session t:
    alpha_{ib,t+1,k} = alpha_{ibt,k} + R_{ibt,k}
```

The saved `belief_prob` is always the pre-session belief.

## Missing Categories

- The category support is the observed support of `i_cat_level1_id` in `processed_data.parquet`.
- Missing/invalid level-1 IDs are mapped to `UNKNOWN_CATEGORY_ID = -124`.
- `UNKNOWN = -124` is kept as a real category.
- Category ID `30` is not added artificially if absent.
- Categories not shown in a session receive count zero but retain prior mass.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
BASE = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed"

INPUT_PATH = BASE / "processed_data.parquet"
OUTPUT_PATH = BASE / "valid_user_belief.parquet"

PRIOR_ALPHA = 1.0
UNKNOWN_CATEGORY_ID = -124
WRITE_OUTPUTS = False

print("input:", INPUT_PATH)
print("output:", OUTPUT_PATH)
print("prior alpha:", PRIOR_ALPHA)


## 1. Load Session-Category Counts

Only exposure/category counts are used. Outcomes such as `watch_ratio`, `play_duration`, and `y_*` are not used in the belief update.


In [ ]:
cols = ["user_id", "burst_id", "session", "time", "video_id", "i_cat_level1_id", "i_cat_level1_name"]
df = pd.read_parquet(INPUT_PATH, columns=cols).copy()
df["time"] = pd.to_datetime(df["time"])
df["i_cat_level1_id"] = pd.to_numeric(df["i_cat_level1_id"], errors="coerce").fillna(UNKNOWN_CATEGORY_ID).astype(int)
df["i_cat_level1_name"] = df["i_cat_level1_name"].astype("string").fillna("UNKNOWN")

cat_table = (
    df[["i_cat_level1_id", "i_cat_level1_name"]]
    .drop_duplicates()
    .sort_values("i_cat_level1_id")
    .reset_index(drop=True)
)
cat_ids = cat_table["i_cat_level1_id"].astype(int).tolist()
cat_to_pos = {cat: pos for pos, cat in enumerate(cat_ids)}

session_counts = (
    df.groupby(["user_id", "burst_id", "session", "i_cat_level1_id"], observed=True)
    .size()
    .rename("n_exposures")
    .reset_index()
)

session_order = (
    df.groupby(["user_id", "burst_id", "session"], observed=True)
    .agg(
        session_start_time=("time", "min"),
        session_end_time=("time", "max"),
        n_interactions=("video_id", "size"),
    )
    .reset_index()
    .sort_values(["user_id", "burst_id", "session_start_time", "session"], kind="mergesort")
)

print("rows:", f"{len(df):,}")
print("sessions:", f"{len(session_order):,}")
print("categories:", len(cat_ids))
display(cat_table)


## 2. Simulate Dirichlet Beliefs

For each user, sessions are processed in chronological burst/session order. Beliefs are saved before updating with the current session's category counts.


In [ ]:
def simulate_user_beliefs(user_sessions, user_counts):
    alpha = np.full(len(cat_ids), PRIOR_ALPHA, dtype="float64")
    rows = []

    counts_by_session = {
        (int(r.burst_id), int(r.session)): (int(r.i_cat_level1_id), float(r.n_exposures))
        for r in user_counts.itertuples(index=False)
    }

    # Easier lookup: session -> dense count vector.
    dense_counts = {}
    for r in user_counts.itertuples(index=False):
        key = (int(r.burst_id), int(r.session))
        if key not in dense_counts:
            dense_counts[key] = np.zeros(len(cat_ids), dtype="float64")
        dense_counts[key][cat_to_pos[int(r.i_cat_level1_id)]] += float(r.n_exposures)

    for sess in user_sessions.itertuples(index=False):
        key = (int(sess.burst_id), int(sess.session))
        belief = alpha / alpha.sum()
        for cat, prob in zip(cat_ids, belief):
            rows.append({
                "user_id": int(sess.user_id),
                "burst_id": int(sess.burst_id),
                "session": int(sess.session),
                "i_cat_level1_id": int(cat),
                "belief_prob": float(prob),
            })
        alpha += dense_counts.get(key, 0.0)

    return rows


belief_rows = []
counts_grouped = {uid: g for uid, g in session_counts.groupby("user_id", observed=True, sort=False)}

for uid, user_sessions in session_order.groupby("user_id", observed=True, sort=False):
    belief_rows.extend(simulate_user_beliefs(user_sessions, counts_grouped.get(uid, session_counts.iloc[0:0])))

belief_df = pd.DataFrame(belief_rows)
belief_df["belief_prob"] = belief_df["belief_prob"].astype("float32")

print("belief rows:", f"{len(belief_df):,}")
display(belief_df.head())


## 3. Validate and Save

Belief probabilities should sum to one for every `(user_id, burst_id, session)`.


In [ ]:
sum_check = (
    belief_df.groupby(["user_id", "burst_id", "session"], observed=True)["belief_prob"]
    .sum()
)
max_abs_error = float((sum_check - 1.0).abs().max())
print("max abs probability-sum error:", max_abs_error)
if max_abs_error > 1e-5:
    raise ValueError("Belief probabilities do not sum to one")

if belief_df["i_cat_level1_id"].eq(UNKNOWN_CATEGORY_ID).any():
    print("UNKNOWN category retained:", UNKNOWN_CATEGORY_ID)

if WRITE_OUTPUTS:
    belief_df.to_parquet(OUTPUT_PATH, index=False)
    print("saved to:", OUTPUT_PATH)
else:
    print("WRITE_OUTPUTS is False; skipped saving.")
